# FinanceBench RAG — Notebook 01: Data + Baseline Retrieval

**Goal of this notebook:** load the FinanceBench questions and filings, split filings into chunks, and measure how often two baseline retrievers (BM25 and dense embeddings) find the page that actually contains the answer.

No GPU and no LLM API needed yet. Generation comes in Notebook 02.

---
## 📘 Concept primer: what is RAG?

An LLM on its own answers from what it memorized during training. It has never seen 3M's 2018 cash flow statement in detail, so it will **hallucinate** (confidently make up) numbers.

**Retrieval-Augmented Generation (RAG)** fixes this with an open-book exam:

1. **Index**: split each document into small pieces (**chunks**) and store them in a searchable form.
2. **Retrieve**: given a question, find the top-*k* most relevant chunks.
3. **Generate**: paste those chunks into the LLM's prompt (the **context**) and ask it to answer *using only that context*.

```
question ──► retriever ──► top-k chunks ──► LLM prompt ──► answer (+ cited sources)
                 ▲
       index of document chunks
```

**Key insight for this project:** if retrieval misses the right page, the LLM can't get the answer right no matter how good it is. So we measure retrieval *separately*, first. That's what this notebook does.

## 0. Setup
Run this once per Colab session. **Use a GPU runtime** (Runtime → Change runtime type → T4 GPU): embedding on CPU takes hours for the full run.

In [1]:
!pip -q install pymupdf rank_bm25 sentence-transformers faiss-cpu

In [2]:
import os, re, json, time, urllib.request
import numpy as np
import pandas as pd
import pymupdf  # PyMuPDF: fast PDF text extraction
from tqdm.auto import tqdm

# Optional: save downloads/indexes to Google Drive so Colab restarts don't wipe them
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/financebench_rag'
else:
    ROOT = '/content/financebench_rag'
PDF_DIR = f'{ROOT}/pdfs'
os.makedirs(PDF_DIR, exist_ok=True)
REPO = 'https://raw.githubusercontent.com/patronus-ai/financebench/main'

Mounted at /content/drive


## 1. Load the benchmark

FinanceBench's open-source split has **150 questions** over **84 filings** (10-Ks, 10-Qs, 8-Ks, earnings releases). Each row has:

- `question`, `answer`: the gold (correct) answer, written by finance analysts
- `doc_name`: which filing the question is about
- `evidence`: list of dicts with `evidence_text` and `evidence_page_num` (**zero-indexed**, so page 0 = first page)
- `question_type`: `metrics-generated` (pull a number from a statement), `domain-relevant`, or `novel-generated`

The evidence page number is gold for us: it is our **ground truth for retrieval**.

In [3]:
qs = pd.read_json(f'{REPO}/data/financebench_open_source.jsonl', lines=True)
docs_meta = pd.read_json(f'{REPO}/data/financebench_document_information.jsonl', lines=True)

# gold pages per question (a question can have >1 evidence page)
qs['gold_pages'] = qs['evidence'].apply(lambda ev: sorted({e['evidence_page_num'] for e in ev}))
print(qs.shape, '| unique docs:', qs.doc_name.nunique())
print(qs.question_type.value_counts())
qs[['doc_name', 'question', 'answer', 'gold_pages']].head()

(150, 12) | unique docs: 84
question_type
metrics-generated    50
domain-relevant      50
novel-generated      50
Name: count, dtype: int64


,doc_name,question,answer,gold_pages
0,3M_2018_10K,What is the FY2018 capital expenditure amount ...,$1577.00,[59]
1,3M_2018_10K,Assume that you are a public equities analyst....,$8.70,[57]
2,3M_2022_10K,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...","[47, 49, 51]"
3,3M_2022_10K,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,[26]
4,3M_2022_10K,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,[24]


## 2. Download the filings and extract text page by page

We keep text **per page** so every chunk remembers which page it came from. That's how we'll check retrieval later.

> **Tip:** For fast iteration, set `PILOT = True` to work with ~10 filings first, then flip it off for the full run.

In [4]:
PILOT = False   # False = full 150-question run; True = quick 10-filing test
doc_names = sorted(qs.doc_name.unique())
if PILOT:
    doc_names = doc_names[:10]
qs_run = qs[qs.doc_name.isin(doc_names)].reset_index(drop=True)
print(len(doc_names), 'docs |', len(qs_run), 'questions')

def get_pdf(name):
    path = f'{PDF_DIR}/{name}.pdf'
    if not os.path.exists(path):
        urllib.request.urlretrieve(f'{REPO}/pdfs/{name}.pdf', path)
    return path

pages = {}  # doc_name -> list of page texts (index = zero-based page number)
for name in tqdm(doc_names):
    with pymupdf.open(get_pdf(name)) as pdf:
        pages[name] = [p.get_text() for p in pdf]
print('total pages:', sum(len(v) for v in pages.values()))

84 docs | 150 questions


  0%|          | 0/84 [00:00<?, ?it/s]

total pages: 12013


## 3. Chunking

📘 **Concept: chunking.** We can't retrieve a whole 150-page 10-K and paste it into a prompt (too long, too slow, too costly). So we split text into chunks.

- **Chunk size** = how many tokens per chunk. *Small* chunks are precise but can cut a table or sentence in half. *Large* chunks keep context but dilute relevance and fill the prompt faster.
- **Overlap** = repeated tokens between neighbouring chunks, so a fact on a boundary isn't lost.
- A **token** ≈ ¾ of an English word. Here we approximate tokens with words to stay simple and fast.

Chunk size is one of our experiment factors (256 / 512 / 1024). Our chunks never cross page boundaries, so each chunk maps to exactly one page.

In [5]:
def chunk_doc(name, page_texts, size=256, overlap=32):
    out = []
    step = size - overlap
    for pno, text in enumerate(page_texts):
        words = text.split()
        if not words:
            continue
        for start in range(0, max(len(words) - overlap, 1), step):
            piece = ' '.join(words[start:start + size])
            out.append({'doc_name': name, 'page': pno, 'text': piece})
    return out

CHUNK_SIZE = 256
chunks = pd.DataFrame([c for n in doc_names for c in chunk_doc(n, pages[n], CHUNK_SIZE)])
print(len(chunks), 'chunks')
chunks.head(3)

31870 chunks


,doc_name,page,text
0,3M_2018_10K,0,low UNITED STATES SECURITIES AND EXCHANGE COMM...
1,3M_2018_10K,0,15(d) of the Act. Yes ☐ No ☒ Indicate by check...
2,3M_2018_10K,0,Exchange Act. Large accelerated filer ☒ Accele...


## 4. Retriever A: BM25 (keyword search)

📘 **Concept: BM25** is the classic search-engine ranking function. It scores a chunk higher when it contains the question's words, especially **rare** words (via *inverse document frequency*, IDF), with diminishing returns for repeats and a penalty for very long chunks:

$$\text{score}(q, d) = \sum_{t \in q} \text{IDF}(t)\cdot\frac{f(t,d)\,(k_1+1)}{f(t,d) + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}$$

where $f(t,d)$ is how often term $t$ appears in chunk $d$. **Strength:** exact matches ("capital expenditure", "FY2018", "EBITDA"). **Weakness:** no notion of synonyms: "capex" ≠ "purchases of property, plant and equipment".

📘 **Concept: per-document retrieval.** Each question names its filing, so we search only that filing's chunks. FinanceBench calls this a *single-store* setup. It's realistic (a user picks the company first) and much cheaper.

In [6]:
from rank_bm25 import BM25Okapi

def tokenize(s):
    return re.findall(r'[a-z0-9]+', s.lower())

bm25_index = {}  # doc_name -> (BM25 object, row indices into `chunks`)
for name, grp in chunks.groupby('doc_name'):
    bm25_index[name] = (BM25Okapi([tokenize(t) for t in grp.text]), grp.index.to_numpy())

def retrieve_bm25(question, doc_name, k=5):
    bm, idx = bm25_index[doc_name]
    scores = bm.get_scores(tokenize(question))
    top = np.argsort(-scores)[:k]
    return idx[top]  # row ids in `chunks`

## 5. Retriever B: dense embeddings + FAISS

📘 **Concept: embeddings.** An embedding model turns text into a vector (here 384 numbers) such that texts with similar **meaning** land close together, even with different words. Retrieval = embed the question, then find the chunk vectors nearest to it.

📘 **Concept: cosine similarity.** "Closeness" is the angle between vectors: $\cos\theta = \frac{u\cdot v}{\|u\|\|v\|}$. If we normalize all vectors to length 1, cosine similarity is just the dot product.

📘 **Concept: FAISS** (Facebook AI Similarity Search) is a library for fast nearest-neighbour search over many vectors. `IndexFlatIP` = exact search by inner product (dot product).

We use **`BAAI/bge-small-en-v1.5`**, a small (33M-parameter) open model that runs fine on CPU. BGE models expect a short instruction prefix on *queries* (not on chunks).

In [7]:
from sentence_transformers import SentenceTransformer
import faiss, torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)   # should say "cuda" if the T4 GPU is attached
EMB_MODEL = 'BAAI/bge-small-en-v1.5'
embedder = SentenceTransformer(EMB_MODEL, device=DEVICE)
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '

# Cache: if embeddings for this exact setup are already on Drive, load them instead of recomputing
tag = 'pilot' if PILOT else 'full'
cache = f'{ROOT}/chunk_vecs_{CHUNK_SIZE}_{tag}.npy'
if os.path.exists(cache) and len(np.load(cache)) == len(chunks):
    chunk_vecs = np.load(cache)
    print('loaded cached embeddings')
else:
    t0 = time.time()
    chunk_vecs = embedder.encode(chunks.text.tolist(), batch_size=128, normalize_embeddings=True,
                                 show_progress_bar=True).astype('float32')
    print(f'embedded {len(chunks)} chunks in {time.time()-t0:.0f}s')
    np.save(cache, chunk_vecs)

dense_index = {}
for name, grp in chunks.groupby('doc_name'):
    ix = faiss.IndexFlatIP(chunk_vecs.shape[1])
    ix.add(chunk_vecs[grp.index.to_numpy()])
    dense_index[name] = (ix, grp.index.to_numpy())

def retrieve_dense(question, doc_name, k=5):
    ix, idx = dense_index[doc_name]
    qv = embedder.encode([QUERY_PREFIX + question], normalize_embeddings=True).astype('float32')
    _, top = ix.search(qv, k)
    return idx[top[0]]

device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/249 [00:00<?, ?it/s]

embedded 31870 chunks in 317s


## 6. Evaluate retrieval

📘 **Concept: Recall@k.** For each question: did *any* of the top-*k* retrieved chunks come from a gold evidence page? Recall@k is the fraction of questions where the answer is yes. It's the single most important RAG metric, because retrieval is the ceiling on answer quality.

📘 **Concept: MRR (Mean Reciprocal Rank).** If the first correct chunk is ranked 1st → score 1, 2nd → ½, 3rd → ⅓, not found → 0; then average. Recall asks *"did we find it?"*; MRR asks *"how high did we rank it?"* This matters because LLMs pay most attention to the top of the context.

In [8]:
def evaluate(retriever, qdf, ks=(1, 3, 5, 10)):
    rows = []
    for _, q in qdf.iterrows():
        got = retriever(q.question, q.doc_name, k=max(ks))
        got_pages = chunks.loc[got, 'page'].tolist()
        hits = [p in q.gold_pages for p in got_pages]
        first = next((i for i, h in enumerate(hits) if h), None)
        r = {f'recall@{k}': any(hits[:k]) for k in ks}
        r['rr'] = 0 if first is None else 1 / (first + 1)
        r['question_type'] = q.question_type
        rows.append(r)
    return pd.DataFrame(rows)

results = {}
for name, fn in [('BM25', retrieve_bm25), ('Dense (bge-small)', retrieve_dense)]:
    res = evaluate(fn, qs_run)
    results[name] = res
summary = pd.DataFrame({n: r.drop(columns='question_type').mean() for n, r in results.items()}).T
summary = summary.rename(columns={'rr': 'MRR'})
summary.round(3)

,recall@1,recall@3,recall@5,recall@10,MRR
BM25,0.147,0.247,0.267,0.307,0.198
Dense (bge-small),0.240,0.447,0.560,0.667,0.375


## 7. Look at failures (do this every time!)

Numbers tell you *how often*; failures tell you *why*. Typical causes on 10-Ks: the answer lives in a **table** that got flattened into a jumble of numbers, or the question uses a synonym of the line item ("capex" vs "purchases of PP&E"). These observations motivate the next experiments (hybrid search, reranking, table-aware chunking).

In [9]:
res = results['Dense (bge-small)']
miss = qs_run[~res['recall@5'].values]
print(len(miss), 'misses at k=5\n')
for _, q in miss.head(3).iterrows():
    print('Q:', q.question[:200])
    print('gold pages:', q.gold_pages, '| answer:', q.answer)
    got = retrieve_dense(q.question, q.doc_name, k=3)
    print('retrieved pages:', chunks.loc[got, 'page'].tolist())
    print('-' * 80)

66 misses at k=5

Q: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
gold pages: [59] | answer: $1577.00
retrieved pages: [46, 42, 38]
--------------------------------------------------------------------------------
Q: Assume that you are a public equities analyst. Answer the following question by primarily using information that is shown in the balance sheet: what is the year end FY2018 net PPNE for 3M? Answer in U
gold pages: [57] | answer: $8.70
retrieved pages: [99, 50, 46]
--------------------------------------------------------------------------------
Q: Is 3M a capital-intensive business based on FY2022 data?
gold pages: [47, 49, 51] | answer: No, the company is managing its CAPEX and Fixed Assets pretty efficiently, which is evident from below key metrics:
CAPEX/Revenue Ratio: 5.1%
Fixed assets/Total Assets: 20%
Return on Assets= 12.4%
retrieved pages: [37, 36, 3]

## Next steps
The baseline above (dense Recall@5 = 0.56 on all 150 questions) is the reference point for the rest of the project.

- **Notebook 02:** hybrid search, a cross-encoder reranker, chunk sizes of 128 / 256 / 384 words, and page-header tagging.
- **Notebook 03a:** the extension, LLM query rewriting.
- **Notebook 03b:** answer generation and accuracy.

Chunk sizes above 384 words were not tested: bge-small reads at most 512 tokens, so larger chunks would be mostly truncated.